In [1]:
import numpy as np
import scipy.stats

import matplotlib.pyplot as plt

In [2]:
title = "# Sampling 1: Exercises"
# Print title and setup TeX defs for both KaTeX and MathJax
import bayesian_stats_course_tools
bayesian_stats_course_tools.misc.display_markdown_and_setup_tex(title)

import matplotlib.style
matplotlib.style.use("bayesian_stats_course_tools.light")

### Exercise

Implement your own rejection sampling routine and test it with different target and proposal distributions.

In [17]:
# true distribution p(x) = x^2 |_ (-3rt(3/2) ^3rt(3/2)
# choose q as 0.5*cos(omega*x) +1,5
plt.figure(figsize=(4, 3))
# Define the interval s.t. p is normalsied
a = (3/2)**(1/3)
x = np.linspace(-a, a, 1000)

# Define the target distribution p(x)
p = x**2 

# Define the proposal distribution q(x)
q = np.cos(12*x) + 3 #omega by choice
#q = np.abs(x)
# Rejection sampling
#M = 1
M=0.5
#
#plot 
plt.figure()
plt.plot(x, p, label="Target distribution p(x)", color = "red")
#plt.plot(x, q, label="Proposal distribution q(x)")
plt.plot(x, M*q, label="Proposal distribution M*q(x)", color = 'green')
plt.xlabel("x")
plt.ylabel("Density")
plt.legend()
plt.title("Target and Proposal Distributions")
plt.show()

# Rejection sampling
samples = []

while len(samples) < 10000:

    #x over intervall s.t. normalsied
    x_new = np.random.uniform(-a, a)
    # evaluate functions
    p_new = x_new**2
    Mq_new = M*(np.cos(12*x_new) + 3)

    # Draw u uniformly between 0 and M*q(x)
    u = np.random.uniform(0, Mq_new)

    # Accept only if the point lies below the target
    if u < p_new:
        samples.append(x_new)


# Plot 2: Histogram of accepted samples and target distribution
plt.figure()
plt.hist(samples, bins=50, density=True, alpha=0.6,
         label="Accepted samples", color = 'purple')
plt.plot(x, p, "r-", label="Target distribution p(x)")
plt.xlabel("x")
plt.ylabel("Density")
plt.legend()
plt.title("Rejection Sampling")
plt.show()

# Plot 3: Residuals (Histogram - Target distribution)
hist, bin_edges = np.histogram(samples, bins=50, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
residuals = hist - bin_centers**2

plt.figure()
plt.axhline(0, color="black", linestyle="--")

plt.plot(x, -p*(M*q-1.5), label="Conspiracy theory: -p*(M*q-1.5)", color = 'green')
plt.bar(bin_centers, residuals, width=np.diff(bin_edges),
        alpha=0.7, color="blue", label="Residuals")
plt.xlabel("x")
plt.ylabel("Residual")
plt.title("Residuals: Histogram - Target Distribution")
plt.legend()
plt.show()

### Exercise

- Implement Metropolis-Hastings for $n$ dimensional distributions
    - Sample from a 2D Gaussian (code on the next slide)
    - Plot the samples in the chain. How do the samples depend on the starting position?
- Show that Metropolis-Hastings satisfies detailed balance
    - Hint: $T(x|y) = q(x|y)\min(1, a)$

In [4]:
# Define variances and correlation
sigma_x = 1
sigma_y = 2
rho = 0.7

# Define mean and covariance
mean = np.array([1, 0.5])
cov = np.array([[sigma_x**2, sigma_x*sigma_y*rho],
                [sigma_x*sigma_y*rho, sigma_y**2]])

# Create distribution object
bivariate_normal = scipy.stats.multivariate_normal(mean=mean, cov=cov)

# Sample 1000 points. Do this with your MCMC implementation instead!
samples = bivariate_normal.rvs(size=1000)

In [5]:
plt.scatter(*samples.T, s=1)
plt.ylabel("$y$")
plt.xlabel("$x$");

In [6]:
#Implement the Metropolis-Hastings algorithm

def metropolis_hastings(n_samples, start, step_size=1):
    # Store all samples
    samples = np.zeros((n_samples, 2))
    
    # Set the initial position
    current = np.array(start, dtype=float)
    
    for i in range(n_samples):
        # Propose a new point near the current point
        proposal = current + np.random.normal(0, step_size, size=2)
    
        # Calculate the acceptance probability
        p_current = bivariate_normal.pdf(current)
        p_proposal = bivariate_normal.pdf(proposal)
        acceptance = min(1, p_proposal / p_current)
    
        # Accept or reject the proposed point
        if np.random.rand() < acceptance:
            current = proposal
    
        # Store the current point
        samples[i] = current

    return samples

In [7]:
# Generate 1000 samples, starting at (0, 0) now with M_S_Method
samples = metropolis_hastings(n_samples=1000, start=[0, 0])

# Plot the samples
plt.scatter(*samples_2.T, s=1, color = 'orange')
plt.scatter(*samples.T, s=1)
plt.ylabel("$y$")
plt.xlabel("$x$")
plt.title("Metropolis-Hastings Samples")
plt.show()

In [ ]:
# Try different starting positions
starts = [[1, 0.5], [5, 5], [-5, 5]]

# Plot the samples for each starting position
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, start in zip(axes, starts):
    # Generate samples from the chosen starting position
    samples_3 = metropolis_hastings(n_samples=1000, start=start)

    # Plot the samples
    ax.scatter(*samples.T, s=1)
    ax.scatter(*samples_3.T, s=1, color='orange')
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")
    ax.set_title(f"Start: {start}")

plt.tight_layout()
plt.show()